In [13]:
import os, subprocess

if not os.path.exists("flyrank-ml-internship"):
    subprocess.run(
        ["git", "clone", "https://github.com/tabeeljohn2/flyrank-ml-internship.git"],
        check=True,
    )

os.chdir("flyrank-ml-internship")
print("Now in:", os.getcwd())

Now in: /content/flyrank-ml-internship/flyrank-ml-internship


In [14]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Base path:", REL)

Connected. Base path: hf://datasets/FlyRank/internship-warehouse


In [15]:
test_q = f"""
SELECT COUNT(*) AS n
FROM read_parquet('{REL}/dim_clients.parquet')
"""
con.sql(test_q).df()

,n
0,104


In [16]:
schema = con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabeeljohn2/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, for one client, on one day.

Table(s) I'll use: dim_content, dim_clients, and
fact_content_daily_performance.

Time window: I develop on one month, month=2026-03, to avoid rate limits
and avoid the sealed final month.

What I'd predict: a proxy — is_declining_label, based on trend direction.
Later I'd want a real future-window label (past 90 days predicting the
next 30 days).

What I exclude: FlyRank's own product scores (like health_score), because
using them as a feature would just copy the existing answer, not discover
anything new.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: impressions, clicks, avg_position, ctr, sessions
Label/proxy: trend_direction (used to build is_declining_label)
Context: content_hash_id, client_hash_id, report_date (for joining/grouping only)
Excluded: health_score, priority_score, action_type (not in our data, and
never to be rebuilt as a feature)

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [19]:
# Query 1: the grain — one row = one content item, one client, one day?
q1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""
print("Query 1 — duplicate grain rows found:")
con.sql(q1).df()

Query 1 — duplicate grain rows found:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [20]:
# Query 2: row count and date span for this month
q2 = f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print("Query 2 — row count and date span:")
con.sql(q2).df()

Query 2 — row count and date span:


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [21]:
# Query 3: availability — how many rows actually have GA4 data?
q3 = f"""
SELECT COUNT(*) AS total,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS with_ga4
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print("Query 3 — availability check:")
con.sql(q3).df()

Query 3 — availability check:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,with_ga4
0,9841378,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Feature availability:
1. gsc_impressions — known the same day it happens (a same-day search count)
2. gsc_clicks — known the same day it happens
3. gsc_avg_position — known the same day, reflects that day's ranking
4. ctr (calculated: gsc_clicks / gsc_impressions) — known the same day,
   built only from that same day's values
5. ga4_sessions — known the same day, a same-day analytics count

In [22]:
# Five features, from the same month
feat_df = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    CASE WHEN gsc_impressions > 0
         THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
         ELSE NULL END AS ctr
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_impressions IS NOT NULL
LIMIT 2000
""").df()
feat_df.head(10)

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,<NA>,0.000000
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,<NA>,0.000000
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,<NA>,0.008000
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,<NA>,0.000000
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,<NA>,0.000000
5,content_36c36abc7650d7af,client_73cda7b4e4f265ea,2026-03-01,239,1,7.347280,<NA>,0.004184
6,content_a7da352b73b02668,client_73cda7b4e4f265ea,2026-03-01,191,0,7.832461,<NA>,0.000000
7,content_05434271b257bb68,client_73cda7b4e4f265ea,2026-03-01,55,0,3.272727,<NA>,0.000000
8,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2026-03-01,77,0,5.636364,<NA>,0.000000
9,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,2026-03-01,2,0,4.500000,<NA>,0.000000


In [23]:
# The leakage trap

# Build a real (simple) proxy label instead of a constant placeholder,
# so the correlation can actually be computed
feat_df["is_declining_label"] = (
    (feat_df["gsc_impressions"] > 10) & (feat_df["gsc_clicks"] == 0)
).astype(int)

print("How many rows are flagged as declining:", feat_df["is_declining_label"].sum(), "out of", len(feat_df))

print("\nBEFORE cheating (honest feature vs label):")
print(feat_df[["ctr", "is_declining_label"]].corr())

# Deliberately leak: build a column directly FROM the label
feat_df["leaky_column"] = feat_df["is_declining_label"] * 100

print("\nAFTER adding a cheat column:")
print(feat_df[["leaky_column", "is_declining_label"]].corr())

feat_df = feat_df.drop(columns=["leaky_column"])
print("\nCheat column removed. Columns left:", feat_df.columns.tolist())

How many rows are flagged as declining: 741 out of 2000

BEFORE cheating (honest feature vs label):
                         ctr  is_declining_label
ctr                 1.000000           -0.079339
is_declining_label -0.079339            1.000000

AFTER adding a cheat column:
                    leaky_column  is_declining_label
leaky_column                 1.0                 1.0
is_declining_label           1.0                 1.0

Cheat column removed. Columns left: ['content_hash_id', 'client_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ctr', 'is_declining_label']


Observed: my honest feature (ctr) showed a weak correlation of -0.079 with
the label, is_declining_label. After adding a column built directly from
the label itself (leaky_column), the correlation jumped to exactly 1.0.
This is the leakage trap: a feature that encodes the answer will always
look perfect, and that perfection is a warning sign, not a good result.
I removed the leaky column and kept only the honest features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.